### 목표 : 도로교통법을 해석하고 판단해주는 ai agent 만들기

### 워크플로우 : LangGraph

### Tool 사용

1. RAG : 도로교통법률 pdf, 판례 txt
2. bm25 : web 서치

### 문서 경로 : ./common_dataset

### RAG 전략

0. 법률, 용어, 판례에 대한 chromaDB 따로 구축
1. 도로교통법률 pdf
    1) 제2조 용어 조항을 분리하여 용어사전을 만들고 keyword search 용으로 사용
    2) chunk 는 글자수로 하지 말고 조항 단위로 분리하여 context 유지. 너무 길 경우 앞의 내용을 압축 요약해서 다음 chunk 에 붙여줌
2. 판례는 항목 별로 chunk 
3. Adaptive RAG : 간단한 법률 검색은 semantic search 와 keyword search 로 node 분기
4. 


### Node 정의

1. 일반적인 생활용어와 법률용어 매칭하는 node
사용자가 사용하는 용어를 구축해둔 용어사전에서 키워드 서치 후, 없는 용어는 다시 semantic search 와 web search 를 통해 매칭 시켜 질문 재생성.


2. 재생성된 질문에 법률을 적용하여 위법 여부를 판단하는 node
판단은 다음과 같은 순서로 강제하고, 사실 판단에 대한 정보가 부족한 경우 사용자로 부터 다시 입력 받아 처음부터 순차적으로 분석되도록 loop 생성
사실 판단 외 내용은 RAG 와 web search 를 통해 보충
    1) 관찰된 사실
    2) 확인되지 않은 사실
    3) 법적 쟁점
    4) 관련 정의
    5) 적용 가능한 법조문
    6) 법률상 구성요건
    7) 각 구성요건별 충족 여부
    8) 예외 규정
    9) 관련 판례
    10) 현재 사실에 대한 적용
    11) 위법 여부
    12) 제재 종류
    13) 불확실성

3. 위법으로 판결날 경우 제재에 대한 node
위법으로 판결날 경우 조치에 대해 판단하여 출력.

4. verifier node
다음 질문 검사:
    1) 인용한 조문이 실제 검색 문서에 존재하는가?
    2) 조문 번호가 정확한가?
    3) 법률과 시행령을 혼동하지 않았는가?
    4) 구성요건을 빠뜨리지 않았는가?
    5) 예외 규정을 검토했는가?
    6) 판례의 결론을 과도하게 일반화하지 않았는가?
    7) 벌금/범칙금/과태료를 혼동하지 않았는가?
    8) 현재 사실만으로 판단 불가능한 것을 단정하지 않았는가?